# CLIQUE Customer Clustering — Training & EDA

This notebook walks through the reproducible pipeline. The heavy lifting lives in
`pipelines/` and `clique/`; here we orchestrate, inspect, and visualise.

- **Synthetic data** (`data/raw/customers_raw.csv`) carries ground-truth segments,
  so it supports supervised metrics (F1 / ROC / AUC).
- **Online Retail II** (real data) has no ground truth, so only intrinsic metrics
  (silhouette, Davies-Bouldin, Calinski-Harabasz) apply.

In [ ]:
# 1. Imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import config
from clique.algorithm import CLIQUE
from pipelines import benchmark, data, preprocess, train
from pipelines.benchmark import run_baseline_comparison
from pipelines.data import build_customer_profiles, clean_data, load_raw_data

sns.set_theme(style="whitegrid")
print("ROOT:", ROOT)

## 2. Generate (or refresh) the synthetic dataset
Produces `data/raw/customers_raw.csv` with a known `true_segment` label.

In [ ]:
data.persist_synthetic()
raw = pd.read_csv(config.RAW_CUSTOMERS_CSV)
raw.head()

## 3. EDA — histograms, boxplots, correlation

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, col in zip(axes.ravel(), config.FEATURE_NAMES):
    raw[col].hist(ax=ax, bins=30)
    ax.set_title(col)
plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 5))
sns.boxplot(data=raw[config.FEATURE_NAMES], orient="h")
plt.title("Feature boxplots (raw scale)"); plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(raw[config.FEATURE_NAMES].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature correlation"); plt.show()

print("Skewness (|skew|>1 => log-transform candidate):")
print(raw[config.FEATURE_NAMES].skew().round(2).to_string())

## 4. Preprocess — log-transform, split (random_state=42), MinMax scale

In [ ]:
prep = preprocess.run_synthetic(test_size=0.2)
X_train, y_train = prep["X_train"], prep["y_train"]
print("Scaled train range:", X_train.min(), X_train.max())

## 5. CLIQUE grid search over (xi, tau)
Selection metric: Adjusted Rand Index (robust to cluster-count inflation).

In [ ]:
grid = train.grid_search(X_train, y_train)
display(grid.sort_values("adjusted_rand", ascending=False))

## 6. Train final model + save artifacts

In [ ]:
model = train.run_synthetic(do_grid_search=True)
print(f"{len(model.clusters_)} clusters across {len(model.subspace_coverage_)} subspaces")

## 7. Evaluate — comparison table + figures (CSV/PNG to results/)

In [ ]:
comparison = benchmark.run_synthetic()
display(comparison)

## 8. (Optional) Real Online Retail II data
Place `online_retail_ii.xlsx` in `data/raw/`. No ground truth => intrinsic metrics only.

In [ ]:
if config.ONLINE_RETAIL_XLSX.exists():
    raw_tx = load_raw_data(str(config.ONLINE_RETAIL_XLSX))
    clean_df, cancellations_df = clean_data(raw_tx, filter_uk=True)
    profiles_real = build_customer_profiles(clean_df, cancellations_df)
    display(profiles_real.describe())
    print("Baseline intrinsic comparison on real profiles:")
    from sklearn.preprocessing import MinMaxScaler
    Xr = MinMaxScaler().fit_transform(np.log1p(profiles_real[config.FEATURE_NAMES].clip(lower=0)))
    display(run_baseline_comparison(Xr))
else:
    print(f"No real dataset at {config.ONLINE_RETAIL_XLSX}; skipping.")